# Multimodal Residual CNN Multi-Task Baseline

This notebook trains a from-scratch multimodal model using RGB images and precomputed log-Mel spectrogram tensors. It does not load FLAC files or compute spectrograms during training.

## 1. Imports and Environment Setup

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score, mean_squared_error, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch import nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from tqdm.auto import tqdm

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def get_device() -> torch.device:
    print('PyTorch:', torch.__version__)
    print('CUDA version:', torch.version.cuda)
    if not torch.cuda.is_available():
        print('CUDA unavailable; using CPU.')
        return torch.device('cpu')
    name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name}')
    print(f'Compute capability: sm_{capability[0]}{capability[1]}')
    print(f'Total VRAM: {total_gb:.2f} GB')
    try:
        _ = (torch.ones(1, device='cuda') + 1).item()
        return torch.device('cuda')
    except Exception as exc:
        print('CUDA is visible but this PyTorch build cannot execute kernels on this GPU.')
        print('Reason:', repr(exc))
        print('For P100, use a PyTorch build compiled with sm_60 support, or switch Kaggle GPU to T4/L4/A100.')
        return torch.device('cpu')


## 2. Configuration

Set paths here. The defaults are placeholders; update them to your Kaggle dataset paths. If CUDA out-of-memory occurs on P100, use `BATCH_SIZE = 16` and `GRADIENT_ACCUMULATION_STEPS = 2`.

In [ ]:
FusionType = Literal['concat', 'gated']
DistanceLossName = Literal['mse', 'smooth_l1']
CropMode = Literal['center', 'random']

@dataclass
class Config:
    SEED: int = 42
    TRAIN_CSV_PATH: Path = Path('/kaggle/input/your-dataset/train.csv')
    TEST_CSV_PATH: Path = Path('/kaggle/input/your-dataset/test.csv')
    IMAGE_DIR: Path = Path('/kaggle/input/your-dataset/images')
    AUDIO_DIR: Path = Path('/kaggle/input/your-dataset/audio')
    MEL_DIR: Path = Path('/kaggle/input/your-dataset/mel')
    CHECKPOINT_DIR: Path = Path('/kaggle/working/checkpoints_residual')
    SUBMISSION_PATH: Path = Path('/kaggle/working/submission.csv')
    IMAGE_COL: str | None = None
    MEL_COL: str | None = None
    SAMPLE_ID_COL: str | None = None
    OBJECT_COL: str | None = None
    DISTANCE_COL: str | None = None
    ZONE_COL: str | None = None
    ILLUMINATION_COL: str | None = None
    MEL_EXTENSIONS: tuple[str, ...] = ('.pt', '.npy')
    PRECOMPUTE_MEL: bool = False
    OVERWRITE_MEL: bool = False
    USE_FULL_AUDIO: bool = True
    AUDIO_SECONDS: float = 2.0  # Used only when USE_FULL_AUDIO=False.
    AUDIO_TARGET_SAMPLE_RATE: int | None = None
    N_FFT: int = 1024
    WIN_LENGTH: int | None = None
    HOP_LENGTH: int = 256
    N_MELS: int = 128
    MEL_F_MIN: float = 0.0
    MEL_F_MAX: float | None = None
    MEL_TIME_FRAMES: int | None = None  # None + USE_FULL_AUDIO=True infers max cached-Mel length.
    MEL_N_MELS: int | None = None
    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 4
    EPOCHS: int = 60
    LEARNING_RATE: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    WARMUP_EPOCHS: int = 3
    IMAGE_EMBED_DIM: int = 256
    AUDIO_EMBED_DIM: int = 256
    FUSION_DIM: int = 256
    DROPOUT: float = 0.30
    EARLY_STOPPING_PATIENCE: int = 10
    GRAD_CLIP_NORM: float = 1.0
    GRADIENT_ACCUMULATION_STEPS: int = 1
    USE_AMP: bool = True
    USE_EMA: bool = False
    EMA_DECAY: float = 0.999
    USE_IMAGE: bool = True
    USE_AUDIO: bool = True
    FUSION_TYPE: FusionType = 'concat'
    DISTANCE_LOSS: DistanceLossName = 'mse'
    USE_DISTANCE_NORMALIZATION: bool = True
    USE_CLASS_WEIGHTS: bool = False
    HORIZONTAL_FLIP: bool = True
    USE_SPEC_AUGMENT: bool = False
    FREQ_MASK_PARAM: int = 8
    TIME_MASK_PARAM: int = 16
    MEL_NOISE_STD: float = 0.01
    RUN_SANITY_CHECKS: bool = True
    OVERFIT_SMALL_SUBSET: bool = False
    OVERFIT_SUBSET_SIZE: int = 64
    OVERFIT_STEPS: int = 20
    RUN_FULL_TRAINING: bool = True

CFG = Config()
seed_everything(CFG.SEED)
DEVICE = get_device()
CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(CFG)


## 3. Dataset Inspection

The notebook auto-detects common column names, then reports label distributions, distance statistics, duplicate identifiers, and missing image/Mel files.

In [ ]:
def require_path(path: Path, description: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f'{description} not found: {path}')

def first_existing_column(df: pd.DataFrame, candidates: list[str], explicit: str | None = None) -> str:
    if explicit is not None:
        if explicit not in df.columns:
            raise KeyError(f'Configured column {explicit!r} not found. Available columns: {list(df.columns)}')
        return explicit
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        key = name.strip().lower()
        if key in normalized:
            return str(normalized[key])
    raise KeyError(f'Could not infer any of {candidates}. Available columns: {list(df.columns)}')

def read_csv(path: Path) -> pd.DataFrame:
    require_path(path, 'CSV')
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

train_df_full = read_csv(CFG.TRAIN_CSV_PATH)
test_df = read_csv(CFG.TEST_CSV_PATH) if CFG.TEST_CSV_PATH.exists() else pd.DataFrame()

IMAGE_COL = first_existing_column(train_df_full, ['image name', 'image_id', 'image', 'filename', 'file_name'], CFG.IMAGE_COL)
MEL_OR_AUDIO_COL = first_existing_column(train_df_full, ['mel name', 'mel', 'mel_path', 'audio name', 'audio_id', 'audio'], CFG.MEL_COL)
SAMPLE_ID_COL = CFG.SAMPLE_ID_COL or IMAGE_COL
OBJECT_COL = first_existing_column(train_df_full, ['Object_Type', 'object_type', 'object'], CFG.OBJECT_COL)
DISTANCE_COL = first_existing_column(train_df_full, ['distance', 'Distance'], CFG.DISTANCE_COL)
ZONE_COL = first_existing_column(train_df_full, ['Location_Zone', 'location_zone', 'zone'], CFG.ZONE_COL)
ILLUMINATION_COL = first_existing_column(train_df_full, ['Illumination', 'illumination', 'light'], CFG.ILLUMINATION_COL)

print('Detected columns:')
print({'image': IMAGE_COL, 'mel_or_audio': MEL_OR_AUDIO_COL, 'sample_id': SAMPLE_ID_COL, 'object': OBJECT_COL, 'distance': DISTANCE_COL, 'zone': ZONE_COL, 'illumination': ILLUMINATION_COL})
print('Training samples:', len(train_df_full))
print('Test samples:', len(test_df))
print('Train CSV columns:', list(train_df_full.columns))
display(train_df_full.head())
display(train_df_full.isna().sum().to_frame('missing_values'))

required_target_cols = [OBJECT_COL, DISTANCE_COL, ZONE_COL, ILLUMINATION_COL]
if train_df_full[required_target_cols].isna().any().any():
    display(train_df_full[train_df_full[required_target_cols].isna().any(axis=1)].head(10))
    raise ValueError('Training CSV contains missing target values.')

print('Duplicate sample identifiers:', int(train_df_full[SAMPLE_ID_COL].duplicated().sum()))
for col in [OBJECT_COL, ZONE_COL, ILLUMINATION_COL]:
    print(f'\n{col} distribution:')
    display(train_df_full[col].value_counts(dropna=False).to_frame('count'))
distance_values = pd.to_numeric(train_df_full[DISTANCE_COL], errors='raise')
print('\nDistance summary:')
display(distance_values.describe().to_frame('distance'))


## Optional: Precompute Log-Mel Spectrogram Cache

Run this section once if your dataset has FLAC files but does not already include `.pt` / `.npy` Mel files. The training Dataset still loads only cached Mel tensors and never computes spectrograms inside `__getitem__`.

In [ ]:
def resolve_audio_path(value: Any) -> Path:
    raw = Path(str(value))
    if raw.is_absolute():
        return raw
    if raw.suffix.lower() == '.flac':
        return CFG.AUDIO_DIR / raw
    return (CFG.AUDIO_DIR / raw).with_suffix('.flac')

def mel_output_path_from_audio_value(value: Any) -> Path:
    raw = Path(str(value))
    if raw.suffix.lower() in CFG.MEL_EXTENSIONS:
        return raw if raw.is_absolute() else CFG.MEL_DIR / raw
    return (CFG.MEL_DIR / raw).with_suffix('.pt')

def crop_or_pad_waveform(waveform: torch.Tensor, sample_rate: int) -> torch.Tensor:
    target_frames = max(1, int(round(CFG.AUDIO_SECONDS * sample_rate)))
    if waveform.size(1) >= target_frames:
        return waveform[:, :target_frames]
    return F.pad(waveform, (0, target_frames - waveform.size(1)))

def compute_log_mel_from_flac(audio_file: Path) -> dict[str, Any]:
    waveform, sample_rate = torchaudio.load(str(audio_file))
    original_sample_rate = sample_rate
    waveform = waveform.mean(dim=0, keepdim=True).float()
    if CFG.AUDIO_TARGET_SAMPLE_RATE is not None and sample_rate != CFG.AUDIO_TARGET_SAMPLE_RATE:
        if CFG.AUDIO_TARGET_SAMPLE_RATE < sample_rate:
            print('Warning: downsampling may remove ultrasonic content:', audio_file.name)
        waveform = torchaudio.functional.resample(waveform, sample_rate, CFG.AUDIO_TARGET_SAMPLE_RATE)
        sample_rate = CFG.AUDIO_TARGET_SAMPLE_RATE
    if not CFG.USE_FULL_AUDIO:
        waveform = crop_or_pad_waveform(waveform, sample_rate)
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=CFG.N_FFT,
        win_length=CFG.WIN_LENGTH,
        hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS,
        f_min=CFG.MEL_F_MIN,
        f_max=CFG.MEL_F_MAX,
        power=2.0,
    )
    mel = mel_transform(waveform)
    mel = torchaudio.transforms.AmplitudeToDB(stype='power')(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return {
        'mel': mel.contiguous().cpu().float(),
        'source_audio': str(audio_file),
        'original_sample_rate': original_sample_rate,
        'sample_rate': sample_rate,
        'n_fft': CFG.N_FFT,
        'win_length': CFG.WIN_LENGTH,
        'hop_length': CFG.HOP_LENGTH,
        'n_mels': CFG.N_MELS,
        'f_min': CFG.MEL_F_MIN,
        'f_max': CFG.MEL_F_MAX,
    }

def precompute_log_mel_cache(df: pd.DataFrame) -> None:
    CFG.MEL_DIR.mkdir(parents=True, exist_ok=True)
    unique_values = sorted(set(df[MEL_OR_AUDIO_COL].astype(str)))
    created = 0
    skipped = 0
    errors: list[dict[str, str]] = []
    for value in tqdm(unique_values, desc='precompute log-mel'):
        out_file = mel_output_path_from_audio_value(value)
        if out_file.exists() and not CFG.OVERWRITE_MEL:
            skipped += 1
            continue
        audio_file = resolve_audio_path(value)
        if not audio_file.exists():
            errors.append({'audio': str(audio_file), 'error': 'missing file'})
            continue
        try:
            payload = compute_log_mel_from_flac(audio_file)
            out_file.parent.mkdir(parents=True, exist_ok=True)
            torch.save(payload, out_file)
            created += 1
        except Exception as exc:
            errors.append({'audio': str(audio_file), 'error': repr(exc)})
    print({'mel_created': created, 'mel_skipped': skipped, 'mel_errors': len(errors), 'mel_dir': str(CFG.MEL_DIR)})
    if errors:
        display(pd.DataFrame(errors).head(20))
        raise RuntimeError('Mel preprocessing failed for one or more audio files.')

if CFG.PRECOMPUTE_MEL:
    require_path(CFG.AUDIO_DIR, 'Audio directory')
    frames = [train_df_full[[MEL_OR_AUDIO_COL]]]
    if len(test_df):
        test_mel_col = first_existing_column(test_df, ['mel name', 'mel', 'mel_path', 'audio name', 'audio_id', 'audio'], CFG.MEL_COL)
        frames.append(test_df[[test_mel_col]].rename(columns={test_mel_col: MEL_OR_AUDIO_COL}))
    precompute_log_mel_cache(pd.concat(frames, ignore_index=True))
else:
    print('Skipping Mel precompute because CFG.PRECOMPUTE_MEL=False. Existing cached Mel files will be used.')


In [ ]:
def resolve_image_path(value: Any) -> Path:
    path = Path(str(value))
    return path if path.is_absolute() else CFG.IMAGE_DIR / path

def resolve_mel_path(value: Any) -> Path:
    raw = Path(str(value))
    if raw.is_absolute() and raw.exists():
        return raw
    candidates: list[Path] = []
    if raw.suffix.lower() in CFG.MEL_EXTENSIONS:
        candidates.append(CFG.MEL_DIR / raw)
    else:
        for ext in CFG.MEL_EXTENSIONS:
            candidates.append((CFG.MEL_DIR / raw).with_suffix(ext))
            candidates.append(CFG.MEL_DIR / f'{raw.name}{ext}')
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

def count_missing_files(df: pd.DataFrame, limit: int | None = None) -> tuple[int, int]:
    subset = df if limit is None else df.head(limit)
    missing_images = sum(not resolve_image_path(v).exists() for v in subset[IMAGE_COL])
    missing_mels = sum(not resolve_mel_path(v).exists() for v in subset[MEL_OR_AUDIO_COL])
    return missing_images, missing_mels

require_path(CFG.IMAGE_DIR, 'Image directory')
require_path(CFG.MEL_DIR, 'Mel directory')
missing_images, missing_mels = count_missing_files(train_df_full)
print({'missing_image_files': missing_images, 'missing_mel_files': missing_mels})
if missing_images or missing_mels:
    bad = []
    for _, row in train_df_full.iterrows():
        ip = resolve_image_path(row[IMAGE_COL]); mp = resolve_mel_path(row[MEL_OR_AUDIO_COL])
        if not ip.exists() or not mp.exists():
            bad.append({'sample': row[SAMPLE_ID_COL], 'image_path': str(ip), 'image_exists': ip.exists(), 'mel_path': str(mp), 'mel_exists': mp.exists()})
        if len(bad) >= 10:
            break
    display(pd.DataFrame(bad))
    raise FileNotFoundError('Missing image or Mel files. Fix paths or column mapping before training.')


## 4. Label Encoding

In [ ]:
def make_encoder(series: pd.Series) -> tuple[dict[str, int], dict[int, str]]:
    classes = sorted(series.astype(str).unique())
    class_to_idx = {label: idx for idx, label in enumerate(classes)}
    idx_to_class = {idx: label for label, idx in class_to_idx.items()}
    return class_to_idx, idx_to_class

object_to_idx, idx_to_object = make_encoder(train_df_full[OBJECT_COL])
zone_to_idx, idx_to_zone = make_encoder(train_df_full[ZONE_COL])
illum_to_idx, idx_to_illum = make_encoder(train_df_full[ILLUMINATION_COL])
encoders = {'object': object_to_idx, 'zone': zone_to_idx, 'illumination': illum_to_idx}
decoders = {'object': idx_to_object, 'zone': idx_to_zone, 'illumination': idx_to_illum}
print({k: len(v) for k, v in encoders.items()})


## 5. Train-Validation Split

In [ ]:
def split_train_valid(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    combined = df[OBJECT_COL].astype(str) + '__' + df[ZONE_COL].astype(str) + '__' + df[ILLUMINATION_COL].astype(str)
    for name, labels in [('combined categorical targets', combined), ('Object_Type', df[OBJECT_COL].astype(str))]:
        try:
            train_df, valid_df = train_test_split(df, test_size=0.2, random_state=CFG.SEED, shuffle=True, stratify=labels)
            print('Split strategy:', name)
            return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), name
        except ValueError as exc:
            print(f'Could not stratify by {name}: {exc}')
    train_df, valid_df = train_test_split(df, test_size=0.2, random_state=CFG.SEED, shuffle=True)
    print('Split strategy: random fallback')
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), 'random fallback'

train_df, valid_df, split_strategy = split_train_valid(train_df_full)
train_distance_mean = float(pd.to_numeric(train_df[DISTANCE_COL]).mean())
train_distance_std = float(pd.to_numeric(train_df[DISTANCE_COL]).std())
if train_distance_std < 1e-8 or math.isnan(train_distance_std):
    train_distance_std = 1.0
print({'train_rows': len(train_df), 'valid_rows': len(valid_df), 'distance_mean': train_distance_mean, 'distance_std': train_distance_std})


## 6. Image Transforms

This baseline uses ImageNet-style RGB normalization for simplicity. Color jitter is mild because illumination is one target.

In [ ]:
RGB_MEAN = [0.485, 0.456, 0.406]
RGB_STD = [0.229, 0.224, 0.225]

train_image_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5 if CFG.HORIZONTAL_FLIP else 0.0),
    transforms.RandomRotation(degrees=6),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.06, hue=0.015),
    transforms.ToTensor(),
    transforms.Normalize(RGB_MEAN, RGB_STD),
])

valid_image_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(RGB_MEAN, RGB_STD),
])


## 7. Multimodal Dataset Class

The Dataset loads RGB images and precomputed `.pt` or `.npy` Mel tensors. It does not load FLAC files and does not compute STFT/Mel features.

In [ ]:
def safe_torch_load(path: Path) -> Any:
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

def load_mel_tensor(path: Path) -> torch.Tensor:
    if not path.exists():
        raise FileNotFoundError(f'Mel file not found: {path}')
    try:
        if path.suffix.lower() == '.pt':
            payload = safe_torch_load(path)
            mel = payload['mel'] if isinstance(payload, dict) and 'mel' in payload else payload
            mel = torch.as_tensor(mel)
        elif path.suffix.lower() == '.npy':
            mel = torch.from_numpy(np.load(path))
        else:
            raise ValueError(f'Unsupported Mel extension: {path.suffix}')
    except Exception as exc:
        raise RuntimeError(f'Failed to load Mel tensor {path}') from exc
    if mel.ndim == 2:
        mel = mel.unsqueeze(0)
    if mel.ndim != 3 or mel.shape[0] != 1:
        raise ValueError(f'Expected Mel shape [n_mels, T] or [1, n_mels, T], got {tuple(mel.shape)} for {path}')
    mel = mel.float()
    if not torch.isfinite(mel).all():
        raise ValueError(f'Mel tensor contains NaN or Inf: {path}')
    return mel

def fix_mel_shape(mel: torch.Tensor, time_frames: int, train: bool) -> torch.Tensor:
    _, n_mels, frames = mel.shape
    if CFG.MEL_N_MELS is not None and n_mels != CFG.MEL_N_MELS:
        mel = F.interpolate(mel.unsqueeze(0), size=(CFG.MEL_N_MELS, frames), mode='bilinear', align_corners=False).squeeze(0)
        n_mels = CFG.MEL_N_MELS
    if frames > time_frames:
        if CFG.USE_FULL_AUDIO:
            raise ValueError(f'Mel has {frames} frames, but MEL_TIME_FRAMES={time_frames}. Re-run inference after including all train/test Mel files or set CFG.MEL_TIME_FRAMES manually.')
        if train:
            start = random.randint(0, frames - time_frames)
        else:
            start = (frames - time_frames) // 2
        mel = mel[:, :, start:start + time_frames]
    elif frames < time_frames:
        mel = F.pad(mel, (0, time_frames - frames))
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    if train and CFG.USE_SPEC_AUGMENT:
        if CFG.FREQ_MASK_PARAM > 0:
            f = random.randint(0, min(CFG.FREQ_MASK_PARAM, mel.shape[1]))
            f0 = random.randint(0, max(0, mel.shape[1] - f))
            mel[:, f0:f0 + f, :] = 0
        if CFG.TIME_MASK_PARAM > 0:
            t = random.randint(0, min(CFG.TIME_MASK_PARAM, mel.shape[2]))
            t0 = random.randint(0, max(0, mel.shape[2] - t))
            mel[:, :, t0:t0 + t] = 0
        if CFG.MEL_NOISE_STD > 0:
            mel = mel + torch.randn_like(mel) * CFG.MEL_NOISE_STD
    return mel

def infer_mel_time_frames(df: pd.DataFrame, max_samples: int = 128) -> int:
    if CFG.MEL_TIME_FRAMES is not None:
        return CFG.MEL_TIME_FRAMES
    scan_df = df if CFG.USE_FULL_AUDIO else df.head(max_samples)
    frame_counts = []
    mel_counts = []
    for value in scan_df[MEL_OR_AUDIO_COL]:
        mel = load_mel_tensor(resolve_mel_path(value))
        mel_counts.append(int(mel.shape[1]))
        frame_counts.append(int(mel.shape[2]))
    inferred = int(max(frame_counts) if CFG.USE_FULL_AUDIO else np.percentile(frame_counts, 90))
    CFG.MEL_N_MELS = CFG.MEL_N_MELS or int(pd.Series(mel_counts).mode().iloc[0])
    print({'inferred_MEL_TIME_FRAMES': inferred, 'inferred_MEL_N_MELS': CFG.MEL_N_MELS, 'full_audio': CFG.USE_FULL_AUDIO})
    return inferred

mel_frame_sources = [train_df_full[[MEL_OR_AUDIO_COL]]]
if len(test_df):
    test_mel_col = first_existing_column(test_df, ['mel name', 'mel', 'mel_path', 'audio name', 'audio_id', 'audio'], CFG.MEL_COL)
    mel_frame_sources.append(test_df[[test_mel_col]].rename(columns={test_mel_col: MEL_OR_AUDIO_COL}))
MEL_TIME_FRAMES = infer_mel_time_frames(pd.concat(mel_frame_sources, ignore_index=True))

class MultimodalDataset(Dataset):
    """Dataset returning RGB image, cached log-Mel tensor, targets, and sample id."""
    def __init__(self, df: pd.DataFrame, image_transform: transforms.Compose, train: bool, has_labels: bool) -> None:
        self.df = df.reset_index(drop=True).copy()
        self.image_transform = image_transform
        self.train = train
        self.has_labels = has_labels
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.iloc[idx]
        image_file = resolve_image_path(row[IMAGE_COL])
        mel_file = resolve_mel_path(row[MEL_OR_AUDIO_COL])
        if not image_file.exists():
            raise FileNotFoundError(f'Missing image file for sample {row[SAMPLE_ID_COL]}: {image_file}')
        image = self.image_transform(Image.open(image_file).convert('RGB'))
        mel = fix_mel_shape(load_mel_tensor(mel_file), MEL_TIME_FRAMES, self.train)
        sample = {'image': image, 'mel': mel, 'sample_id': str(row[SAMPLE_ID_COL])}
        if self.has_labels:
            distance = float(row[DISTANCE_COL])
            if CFG.USE_DISTANCE_NORMALIZATION:
                distance_target = (distance - train_distance_mean) / train_distance_std
            else:
                distance_target = distance
            sample.update({
                'object_target': torch.tensor(object_to_idx[str(row[OBJECT_COL])], dtype=torch.long),
                'distance_target': torch.tensor(distance_target, dtype=torch.float32),
                'distance_raw': torch.tensor(distance, dtype=torch.float32),
                'zone_target': torch.tensor(zone_to_idx[str(row[ZONE_COL])], dtype=torch.long),
                'illumination_target': torch.tensor(illum_to_idx[str(row[ILLUMINATION_COL])], dtype=torch.long),
            })
        return sample

train_dataset = MultimodalDataset(train_df, train_image_transform, train=True, has_labels=True)
valid_dataset = MultimodalDataset(valid_df, valid_image_transform, train=False, has_labels=True)
test_dataset = MultimodalDataset(test_df, valid_image_transform, train=False, has_labels=False) if len(test_df) else None

sample = train_dataset[0]
print({k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in sample.items() if k != 'sample_id'})
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
img = sample['image'].permute(1, 2, 0).numpy() * np.array(RGB_STD) + np.array(RGB_MEAN)
axes[0].imshow(np.clip(img, 0, 1)); axes[0].set_title('RGB image'); axes[0].axis('off')
axes[1].imshow(sample['mel'][0].numpy(), aspect='auto', origin='lower'); axes[1].set_title('Cached Mel'); plt.show()


## 8. Residual Block Implementation

In [ ]:
class ResidualBlock(nn.Module):
    """2D residual block with optional channel change and stride-2 downsampling."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.activation = nn.SiLU(inplace=True)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.main(x) + self.shortcut(x))


## 9. Image Residual Backbone

In [ ]:
class ImageResidualEncoder(nn.Module):
    """From-scratch residual CNN for RGB images."""
    def __init__(self, embed_dim: int, dropout: float) -> None:
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True))
        self.stage1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.stage2 = nn.Sequential(ResidualBlock(32, 64, stride=2), ResidualBlock(64, 64))
        self.stage3 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.stage4 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.SiLU(inplace=True), nn.Dropout(dropout))
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        x = self.stem(image)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x); x = self.stage4(x)
        return self.proj(self.pool(x))


## 10. Audio Residual Backbone

In [ ]:
class AudioResidualEncoder(nn.Module):
    """From-scratch residual CNN treating log-Mel as a single-channel image."""
    def __init__(self, embed_dim: int, dropout: float) -> None:
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.SiLU(inplace=True))
        self.stage1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.stage2 = nn.Sequential(ResidualBlock(32, 64, stride=2), ResidualBlock(64, 64))
        self.stage3 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.stage4 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.SiLU(inplace=True), nn.Dropout(dropout))
    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x = self.stem(mel)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x); x = self.stage4(x)
        return self.proj(self.pool(x))


## 11. Multimodal Fusion Module

In [ ]:
class ConcatenationFusion(nn.Module):
    def __init__(self, input_dim: int, fusion_dim: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.SiLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim), nn.SiLU(inplace=True),
        )
    def forward(self, features: list[torch.Tensor]) -> torch.Tensor:
        return self.net(torch.cat(features, dim=1))

class GatedFusion(nn.Module):
    def __init__(self, image_dim: int, audio_dim: int, fusion_dim: int, dropout: float) -> None:
        super().__init__()
        self.image_proj = nn.Linear(image_dim, fusion_dim)
        self.audio_proj = nn.Linear(audio_dim, fusion_dim)
        self.gate = nn.Sequential(nn.Linear(image_dim + audio_dim, fusion_dim), nn.Sigmoid())
        self.out = nn.Sequential(nn.LayerNorm(fusion_dim), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(fusion_dim, fusion_dim), nn.SiLU(inplace=True))
    def forward(self, features: list[torch.Tensor]) -> torch.Tensor:
        image_feat, audio_feat = features
        gate = self.gate(torch.cat([image_feat, audio_feat], dim=1))
        fused = gate * self.image_proj(image_feat) + (1.0 - gate) * self.audio_proj(audio_feat)
        return self.out(fused)


## 12. Four Task-Specific Heads

In [ ]:
def make_head(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:
    return nn.Sequential(nn.Linear(in_dim, 128), nn.SiLU(inplace=True), nn.Dropout(dropout), nn.Linear(128, out_dim))


## 13. Complete Multimodal Model

In [ ]:
class MultimodalMultiTaskModel(nn.Module):
    """Residual multimodal model with configurable RGB/audio ablations."""
    def __init__(self, cfg: Config, num_object: int, num_zone: int, num_illum: int) -> None:
        super().__init__()
        if not cfg.USE_IMAGE and not cfg.USE_AUDIO:
            raise ValueError('At least one modality must be enabled.')
        self.cfg = cfg
        self.image_encoder = ImageResidualEncoder(cfg.IMAGE_EMBED_DIM, cfg.DROPOUT) if cfg.USE_IMAGE else None
        self.audio_encoder = AudioResidualEncoder(cfg.AUDIO_EMBED_DIM, cfg.DROPOUT) if cfg.USE_AUDIO else None
        if cfg.USE_IMAGE and cfg.USE_AUDIO:
            if cfg.FUSION_TYPE == 'concat':
                self.fusion = ConcatenationFusion(cfg.IMAGE_EMBED_DIM + cfg.AUDIO_EMBED_DIM, cfg.FUSION_DIM, cfg.DROPOUT)
            elif cfg.FUSION_TYPE == 'gated':
                self.fusion = GatedFusion(cfg.IMAGE_EMBED_DIM, cfg.AUDIO_EMBED_DIM, cfg.FUSION_DIM, cfg.DROPOUT)
            else:
                raise ValueError(f'Unsupported fusion: {cfg.FUSION_TYPE}')
            fused_dim = cfg.FUSION_DIM
        elif cfg.USE_IMAGE:
            self.fusion = nn.Identity(); fused_dim = cfg.IMAGE_EMBED_DIM
        else:
            self.fusion = nn.Identity(); fused_dim = cfg.AUDIO_EMBED_DIM
        self.object_head = make_head(fused_dim, num_object, cfg.DROPOUT)
        self.distance_head = make_head(fused_dim, 1, cfg.DROPOUT)
        self.zone_head = make_head(fused_dim, num_zone, cfg.DROPOUT)
        self.illumination_head = make_head(fused_dim, num_illum, cfg.DROPOUT)
        self.apply(self._init_weights)
    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
    def extract_features(self, image: torch.Tensor, mel: torch.Tensor) -> torch.Tensor:
        feats: list[torch.Tensor] = []
        if self.image_encoder is not None:
            feats.append(self.image_encoder(image))
        if self.audio_encoder is not None:
            feats.append(self.audio_encoder(mel))
        if len(feats) == 1:
            return feats[0]
        return self.fusion(feats)
    def forward(self, image: torch.Tensor, mel: torch.Tensor) -> dict[str, torch.Tensor]:
        features = self.extract_features(image, mel)
        return {
            'object_logits': self.object_head(features),
            'distance_pred': self.distance_head(features).squeeze(-1),
            'zone_logits': self.zone_head(features),
            'illumination_logits': self.illumination_head(features),
        }

model = MultimodalMultiTaskModel(CFG, len(object_to_idx), len(zone_to_idx), len(illum_to_idx)).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {num_params:,}')
print(f'Approx model size: {num_params * 4 / 1024**2:.2f} MB')


## 14. Loss Functions

In [ ]:
def class_weights_for(series: pd.Series, encoder: dict[str, int]) -> torch.Tensor | None:
    if not CFG.USE_CLASS_WEIGHTS:
        return None
    labels = torch.tensor([encoder[str(x)] for x in series], dtype=torch.long)
    counts = torch.bincount(labels, minlength=len(encoder)).float().clamp_min(1.0)
    return (counts.sum() / (len(encoder) * counts)).to(DEVICE)

loss_fns = {
    'object': nn.CrossEntropyLoss(weight=class_weights_for(train_df[OBJECT_COL], object_to_idx)),
    'zone': nn.CrossEntropyLoss(weight=class_weights_for(train_df[ZONE_COL], zone_to_idx)),
    'illumination': nn.CrossEntropyLoss(weight=class_weights_for(train_df[ILLUMINATION_COL], illum_to_idx)),
    'distance': nn.MSELoss() if CFG.DISTANCE_LOSS == 'mse' else nn.SmoothL1Loss(),
}

def compute_multitask_loss(outputs: dict[str, torch.Tensor], batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, dict[str, float]]:
    object_loss = loss_fns['object'](outputs['object_logits'], batch['object_target'])
    distance_loss = loss_fns['distance'](outputs['distance_pred'], batch['distance_target'])
    zone_loss = loss_fns['zone'](outputs['zone_logits'], batch['zone_target'])
    illum_loss = loss_fns['illumination'](outputs['illumination_logits'], batch['illumination_target'])
    weighted_object = 0.40 * object_loss
    weighted_distance = 0.30 * distance_loss
    weighted_zone = 0.20 * zone_loss
    weighted_illum = 0.10 * illum_loss
    total = weighted_object + weighted_distance + weighted_zone + weighted_illum
    logs = {
        'object_loss': float(object_loss.detach().cpu()), 'distance_loss': float(distance_loss.detach().cpu()),
        'zone_loss': float(zone_loss.detach().cpu()), 'illumination_loss': float(illum_loss.detach().cpu()),
        'weighted_object_loss': float(weighted_object.detach().cpu()), 'weighted_distance_loss': float(weighted_distance.detach().cpu()),
        'weighted_zone_loss': float(weighted_zone.detach().cpu()), 'weighted_illumination_loss': float(weighted_illum.detach().cpu()),
        'total_loss': float(total.detach().cpu()),
    }
    return total, logs


## 15. Competition Metric

In [ ]:
def denormalize_distance(values: np.ndarray) -> np.ndarray:
    values = values.astype(np.float32)
    if CFG.USE_DISTANCE_NORMALIZATION:
        return values * np.float32(train_distance_std) + np.float32(train_distance_mean)
    return values

def compute_competition_metrics(y_true: dict[str, np.ndarray], y_pred: dict[str, np.ndarray]) -> dict[str, float]:
    distance_pred = denormalize_distance(y_pred['distance'])
    rmse = float(np.sqrt(mean_squared_error(y_true['distance_raw'], distance_pred)))
    distance_component = max(0.0, 1.0 - rmse / 2.0)
    object_f1 = float(f1_score(y_true['object'], y_pred['object'], average='macro', zero_division=0))
    zone_f1 = float(f1_score(y_true['zone'], y_pred['zone'], average='macro', zero_division=0))
    illum_f1 = float(f1_score(y_true['illumination'], y_pred['illumination'], average='macro', zero_division=0))
    score = 100.0 * (0.40 * object_f1 + 0.30 * distance_component + 0.20 * zone_f1 + 0.10 * illum_f1)
    return {'object_macro_f1': object_f1, 'distance_rmse': rmse, 'distance_component': distance_component, 'zone_macro_f1': zone_f1, 'illumination_macro_f1': illum_f1, 'competition_score': score}


## 16-18. Training, Validation, and Best Checkpoint Saving

In [ ]:
def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        persistent_workers=(CFG.NUM_WORKERS > 0),
        drop_last=False,
    )

train_loader = make_loader(train_dataset, shuffle=True)
valid_loader = make_loader(valid_dataset, shuffle=False)

def batch_to_device(batch: dict[str, Any]) -> dict[str, Any]:
    moved: dict[str, Any] = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            moved[key] = value.to(DEVICE, non_blocking=True)
        else:
            moved[key] = value
    return moved

def make_scheduler(optimizer: torch.optim.Optimizer, steps_per_epoch: int) -> LambdaLR:
    total_steps = max(1, steps_per_epoch * CFG.EPOCHS)
    warmup_steps = max(0, min(total_steps - 1, steps_per_epoch * CFG.WARMUP_EPOCHS))
    def fn(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    return LambdaLR(optimizer, fn)

optimizer = AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
scheduler = make_scheduler(optimizer, len(train_loader))
scaler = GradScaler(enabled=(CFG.USE_AMP and DEVICE.type == 'cuda'))

def save_checkpoint(path: Path, epoch: int, best_score: float, history: list[dict[str, Any]]) -> None:
    payload = {
        'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(), 'epoch': epoch, 'best_competition_score': best_score,
        'config': asdict(CFG), 'encoders': encoders, 'decoders': {k: {str(i): v for i, v in d.items()} for k, d in decoders.items()},
        'distance_stats': {'mean': train_distance_mean, 'std': train_distance_std, 'normalized': CFG.USE_DISTANCE_NORMALIZATION},
        'columns': {'image': IMAGE_COL, 'mel_or_audio': MEL_OR_AUDIO_COL, 'sample_id': SAMPLE_ID_COL, 'object': OBJECT_COL, 'distance': DISTANCE_COL, 'zone': ZONE_COL, 'illumination': ILLUMINATION_COL},
        'mel_time_frames': MEL_TIME_FRAMES, 'mel_n_mels': CFG.MEL_N_MELS, 'history': history,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)

def train_one_epoch(epoch: int) -> dict[str, float]:
    model.train()
    totals: dict[str, float] = {}
    seen = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'train {epoch}', leave=False)
    for step, batch in enumerate(pbar, start=1):
        batch = batch_to_device(batch)
        with autocast(enabled=(CFG.USE_AMP and DEVICE.type == 'cuda')):
            outputs = model(batch['image'], batch['mel'])
            loss, logs = compute_multitask_loss(outputs, batch)
            loss = loss / CFG.GRADIENT_ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if step % CFG.GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            if CFG.GRAD_CLIP_NORM > 0:
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        batch_size = batch['image'].size(0); seen += batch_size
        for key, value in logs.items(): totals[key] = totals.get(key, 0.0) + value * batch_size
        pbar.set_postfix(loss=logs['total_loss'], lr=optimizer.param_groups[0]['lr'])
    return {f'train_{k}': v / max(1, seen) for k, v in totals.items()}

@torch.no_grad()
def validate_one_epoch() -> tuple[dict[str, float], dict[str, np.ndarray], dict[str, np.ndarray]]:
    model.eval()
    totals: dict[str, float] = {}; seen = 0
    yt = {'object': [], 'zone': [], 'illumination': [], 'distance_raw': []}
    yp = {'object': [], 'zone': [], 'illumination': [], 'distance': []}
    for batch in tqdm(valid_loader, desc='valid', leave=False):
        batch = batch_to_device(batch)
        with autocast(enabled=(CFG.USE_AMP and DEVICE.type == 'cuda')):
            outputs = model(batch['image'], batch['mel'])
            loss, logs = compute_multitask_loss(outputs, batch)
        batch_size = batch['image'].size(0); seen += batch_size
        for key, value in logs.items(): totals[key] = totals.get(key, 0.0) + value * batch_size
        yt['object'].append(batch['object_target'].detach().cpu().numpy()); yp['object'].append(outputs['object_logits'].argmax(1).detach().cpu().numpy())
        yt['zone'].append(batch['zone_target'].detach().cpu().numpy()); yp['zone'].append(outputs['zone_logits'].argmax(1).detach().cpu().numpy())
        yt['illumination'].append(batch['illumination_target'].detach().cpu().numpy()); yp['illumination'].append(outputs['illumination_logits'].argmax(1).detach().cpu().numpy())
        yt['distance_raw'].append(batch['distance_raw'].detach().cpu().numpy()); yp['distance'].append(outputs['distance_pred'].detach().cpu().numpy())
    y_true = {k: np.concatenate(v) for k, v in yt.items()}
    y_pred = {k: np.concatenate(v) for k, v in yp.items()}
    metrics = compute_competition_metrics(y_true, y_pred)
    avg_losses = {f'val_{k}': v / max(1, seen) for k, v in totals.items()}
    return {**avg_losses, **metrics}, y_true, y_pred


## Sanity Checks

In [ ]:
def run_sanity_checks() -> None:
    batch = next(iter(train_loader))
    print('Sample tensor shapes:', {k: tuple(v.shape) for k, v in train_dataset[0].items() if torch.is_tensor(v)})
    print('Batch tensor shapes:', {k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})
    batch = batch_to_device(batch)
    model.train()
    outputs = model(batch['image'], batch['mel'])
    print('Output shapes:', {k: tuple(v.shape) for k, v in outputs.items()})
    loss, logs = compute_multitask_loss(outputs, batch)
    print(logs)
    assert torch.isfinite(loss), 'Total loss is not finite.'
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_exists = any(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
    assert grad_exists, 'No finite gradients found.'
    optimizer.zero_grad(set_to_none=True)
    print('Sanity checks passed.')

if CFG.RUN_SANITY_CHECKS:
    run_sanity_checks()


## Optional Small-Subset Overfit Check

In [ ]:
def overfit_small_subset() -> None:
    subset = Subset(train_dataset, list(range(min(CFG.OVERFIT_SUBSET_SIZE, len(train_dataset)))))
    loader = DataLoader(subset, batch_size=min(16, CFG.BATCH_SIZE), shuffle=True, num_workers=0)
    tmp_model = MultimodalMultiTaskModel(CFG, len(object_to_idx), len(zone_to_idx), len(illum_to_idx)).to(DEVICE)
    tmp_opt = AdamW(tmp_model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
    losses = []
    for step, batch in enumerate(loader):
        if step >= CFG.OVERFIT_STEPS: break
        batch = batch_to_device(batch)
        tmp_opt.zero_grad(set_to_none=True)
        out = tmp_model(batch['image'], batch['mel'])
        loss, _ = compute_multitask_loss(out, batch)
        loss.backward(); tmp_opt.step(); losses.append(float(loss.detach().cpu()))
    print('Small-subset losses:', losses[:5], '...', losses[-5:])

if CFG.OVERFIT_SMALL_SUBSET:
    overfit_small_subset()


## Full Training

In [ ]:
history: list[dict[str, Any]] = []
best_score = -float('inf')
best_epoch = 0
epochs_without_improvement = 0
best_y_true: dict[str, np.ndarray] | None = None
best_y_pred: dict[str, np.ndarray] | None = None

if CFG.RUN_FULL_TRAINING:
    for epoch in range(1, CFG.EPOCHS + 1):
        train_stats = train_one_epoch(epoch)
        val_stats, y_true, y_pred = validate_one_epoch()
        row = {'epoch': epoch, 'lr': optimizer.param_groups[0]['lr'], **train_stats, **val_stats}
        history.append(row)
        pd.DataFrame(history).to_csv(CFG.CHECKPOINT_DIR / 'training_history.csv', index=False)
        print(f"epoch {epoch:03d} | train_loss {row['train_total_loss']:.4f} | val_loss {row['val_total_loss']:.4f} | obj_f1 {row['object_macro_f1']:.4f} | zone_f1 {row['zone_macro_f1']:.4f} | illum_f1 {row['illumination_macro_f1']:.4f} | rmse {row['distance_rmse']:.4f} | score {row['competition_score']:.4f}")
        save_checkpoint(CFG.CHECKPOINT_DIR / 'last_model.pt', epoch, max(best_score, row['competition_score']), history)
        if row['competition_score'] > best_score:
            best_score = float(row['competition_score']); best_epoch = epoch; best_y_true = y_true; best_y_pred = y_pred
            save_checkpoint(CFG.CHECKPOINT_DIR / 'best_model.pt', epoch, best_score, history)
            epochs_without_improvement = 0
            print('  saved new best checkpoint')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= CFG.EARLY_STOPPING_PATIENCE:
                print('Early stopping triggered.')
                break
print({'best_epoch': best_epoch, 'best_competition_score': best_score})


## 19. Learning Curves

In [ ]:
history_df = pd.DataFrame(history)
def plot_metric(columns: list[str], title: str) -> None:
    if history_df.empty: return
    history_df.plot(x='epoch', y=columns, figsize=(8, 4), marker='o')
    plt.title(title); plt.grid(True); plt.show()

plot_metric(['train_total_loss', 'val_total_loss'], 'Total loss')
plot_metric(['object_macro_f1'], 'Object Macro F1')
plot_metric(['distance_rmse'], 'Distance RMSE')
plot_metric(['zone_macro_f1'], 'Zone Macro F1')
plot_metric(['illumination_macro_f1'], 'Illumination Macro F1')
plot_metric(['competition_score'], 'Competition score')
plot_metric(['lr'], 'Learning rate')


## 20. Final Validation Evaluation

In [ ]:
def load_checkpoint(path: Path) -> dict[str, Any]:
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    return checkpoint

best_checkpoint_path = CFG.CHECKPOINT_DIR / 'best_model.pt'
if best_checkpoint_path.exists():
    checkpoint = load_checkpoint(best_checkpoint_path)
    val_stats, final_y_true, final_y_pred = validate_one_epoch()
    print('Best epoch:', checkpoint['epoch'])
    print('Best checkpoint score:', checkpoint['best_competition_score'])
    print(val_stats)
    print(f'Trainable parameters: {num_params:,}')
    print(f'Approx model size: {num_params * 4 / 1024**2:.2f} MB')
    for task, labels in [('object', idx_to_object), ('zone', idx_to_zone), ('illumination', idx_to_illum)]:
        cm = confusion_matrix(final_y_true[task], final_y_pred[task], labels=list(labels.keys()))
        ConfusionMatrixDisplay(cm, display_labels=[labels[i] for i in labels]).plot(xticks_rotation=45)
        plt.title(f'{task} confusion matrix'); plt.show()
        p, r, f, s = precision_recall_fscore_support(final_y_true[task], final_y_pred[task], labels=list(labels.keys()), zero_division=0)
        display(pd.DataFrame({'task': task, 'class': [labels[i] for i in labels], 'precision': p, 'recall': r, 'f1': f, 'support': s}))
    pred_distance_raw = denormalize_distance(final_y_pred['distance'])
    plt.figure(figsize=(5, 5)); plt.scatter(final_y_true['distance_raw'], pred_distance_raw, s=10, alpha=0.6); plt.xlabel('True distance'); plt.ylabel('Predicted distance'); plt.title('True vs predicted distance'); plt.grid(True); plt.show()
    residuals = pred_distance_raw - final_y_true['distance_raw']
    plt.hist(residuals, bins=30); plt.title('Distance residuals'); plt.show()
else:
    print('No best checkpoint found yet.')


## 21. Inference and Submission Generation

In [ ]:
@torch.no_grad()
def predict_test() -> pd.DataFrame:
    if test_dataset is None:
        raise ValueError('No test CSV loaded.')
    load_checkpoint(CFG.CHECKPOINT_DIR / 'best_model.pt')
    model.eval()
    loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'), persistent_workers=(CFG.NUM_WORKERS > 0))
    rows: list[dict[str, Any]] = []
    for batch in tqdm(loader, desc='predict'):
        batch = batch_to_device(batch)
        outputs = model(batch['image'], batch['mel'])
        object_idx = outputs['object_logits'].argmax(1).detach().cpu().numpy()
        zone_idx = outputs['zone_logits'].argmax(1).detach().cpu().numpy()
        illum_idx = outputs['illumination_logits'].argmax(1).detach().cpu().numpy()
        distance = denormalize_distance(outputs['distance_pred'].detach().cpu().numpy())
        for sample_id, oi, zi, ii, dist in zip(batch['sample_id'], object_idx, zone_idx, illum_idx, distance):
            rows.append({SAMPLE_ID_COL: sample_id, OBJECT_COL: idx_to_object[int(oi)], DISTANCE_COL: float(dist), ZONE_COL: idx_to_zone[int(zi)], ILLUMINATION_COL: idx_to_illum[int(ii)]})
    return pd.DataFrame(rows)

def create_submission() -> pd.DataFrame:
    submission = predict_test()
    CFG.SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(CFG.SUBMISSION_PATH, index=False)
    print('Submission shape:', submission.shape)
    print('Columns:', list(submission.columns))
    print('Missing values:', submission.isna().sum().to_dict())
    display(submission.head())
    print('Saved to:', CFG.SUBMISSION_PATH)
    return submission

if test_dataset is not None and (CFG.CHECKPOINT_DIR / 'best_model.pt').exists():
    submission_df = create_submission()


## 22. Optional Ablation Switches

Run ablations by changing the configuration cell and rerunning from the model creation cell onward:

- RGB only: `USE_IMAGE=True`, `USE_AUDIO=False`
- Mel only: `USE_IMAGE=False`, `USE_AUDIO=True`
- RGB + Mel: `USE_IMAGE=True`, `USE_AUDIO=True`
- Concatenation fusion: `FUSION_TYPE='concat'`
- Gated fusion: `FUSION_TYPE='gated'`
- MSE vs SmoothL1: `DISTANCE_LOSS='mse'` or `'smooth_l1'`
- Distance normalization on/off: `USE_DISTANCE_NORMALIZATION=True/False`
- Class weights on/off: `USE_CLASS_WEIGHTS=True/False`